# LeBron James Shot Analytics - Final Results

Summary output from all four mart tables: era analysis, tenure performance, clutch gene, and opponent analysis.

In [ ]:
import duckdb
from pathlib import Path

db_path = Path.cwd().parent / 'lebron_analytics.duckdb'
conn = duckdb.connect(str(db_path), read_only=True)
print(f'Connected: {db_path}')

## 1. Era Analysis: Dead-ball vs Three-point Era

In [ ]:
era_summary = conn.execute("""
	SELECT
		era, total_shots, total_seasons, overall_fg_pct,
		pct_shots_that_are_2pt, pct_shots_that_are_3pt,
		fg2_pct, fg3_pct, avg_shot_distance, points_per_shot
	FROM main.mart_era_analysis
	GROUP BY
		era, total_shots, total_seasons, overall_fg_pct,
		pct_shots_that_are_2pt, pct_shots_that_are_3pt,
		fg2_pct, fg3_pct, avg_shot_distance, points_per_shot
	ORDER BY era
""").fetchall()

print(f"{'Era':<20} {'Shots':>7} {'Seasons':>8} {'FG%':>6} {'2PT%':>6} {'3PT%':>6} {'FG2%':>6} {'FG3%':>6} {'Dist':>6} {'PPS':>6}")
print('-' * 80)
for era, shots, seasons, fg, pct2, pct3, fg2, fg3, dist, pps in era_summary:
	print(f'{era:<20} {shots:>7,} {seasons:>8} {fg:>6}% {pct2:>5}% {pct3:>5}% {fg2:>5}% {fg3:>5}% {dist:>5}ft {pps:>6}')

In [ ]:
print('Season-by-season 3PT rate trend:')
season_trend = conn.execute("""
	SELECT era, season, season_three_pt_rate, season_fg_pct, season_avg_shot_distance
	FROM main.mart_era_analysis
	GROUP BY era, season, season_three_pt_rate, season_fg_pct, season_avg_shot_distance
	ORDER BY season
""").fetchall()

for era, season, rate3, fg, dist in season_trend:
	bar = '#' * int((rate3 or 0) / 2)
	print(f'  {season} [{era[:4]}]  3PT%: {rate3:>5}%  {bar:<25}  FG: {fg}%  Dist: {dist}ft')

## 2. Tenure Performance: Cavs I -> Heat -> Cavs II -> Lakers

In [ ]:
tenure_summary = conn.execute("""
	SELECT
		tenure, tenure_order, first_season, last_season,
		total_shots, total_games, overall_fg_pct, three_pt_rate,
		fg2_pct, fg3_pct, shots_per_game, home_fg_pct, away_fg_pct
	FROM main.mart_tenure_performance
	GROUP BY
		tenure, tenure_order, first_season, last_season,
		total_shots, total_games, overall_fg_pct, three_pt_rate,
		fg2_pct, fg3_pct, shots_per_game, home_fg_pct, away_fg_pct
	ORDER BY tenure_order
""").fetchall()

print(f"{'Tenure':<10} {'Seasons':<20} {'Shots':>7} {'Games':>6} {'FG%':>6} {'3PT%':>6} {'FG2%':>6} {'FG3%':>6} {'Home':>6} {'Away':>6}")
print('-' * 80)
for t, order, first, last, shots, games, fg, rate3, fg2, fg3, spg, home, away in tenure_summary:
	print(f'{t:<10} {first}-{last}  {shots:>7,} {games:>6,} {fg:>6}% {rate3:>5}% {fg2:>5}% {fg3:>5}% {home:>5}% {away:>5}%')

## 3. The Clutch Gene: Regular vs Clutch Performance

In [ ]:
clutch = conn.execute("""
	SELECT
		situation, total_shots, fg_pct, three_pt_rate, points_per_shot,
		critical_time_shots, critical_fg_pct,
		final_possession_shots, final_possession_fg_pct,
		cavs1_shots, cavs1_fg_pct,
		heat_shots,  heat_fg_pct,
		cavs2_shots, cavs2_fg_pct,
		lakers_shots, lakers_fg_pct
	FROM main.mart_clutch_gene
	ORDER BY is_clutch_time DESC
""").fetchall()

print(f"{'Situation':<10} {'Shots':>7} {'FG%':>6} {'3PT%':>6} {'PPS':>6}  | Cavs1  Heat  CavsII Lakers")
print('-' * 80)
for row in clutch:
	sit, shots, fg, r3, pps, crit_s, crit_fg, fp_s, fp_fg, c1s, c1fg, hs, hfg, c2s, c2fg, ls, lfg = row
	print(f'{sit:<10} {shots:>7,} {fg:>6}% {r3:>5}% {pps:>6}  | {c1fg or "N/A":>5}% {hfg or "N/A":>5}% {c2fg or "N/A":>6}% {lfg or "N/A":>5}%')

print(f'\n  Critical time (<2min): {clutch[0][5] if clutch else 0:,} shots, {clutch[0][6] if clutch else 0}% FG')
print(f'  Final possessions:     {clutch[0][7] if clutch else 0:,} shots, {clutch[0][8] if clutch else 0}% FG (game-winning)')

## 4. Opponent Analysis: Top 10 Most-Faced Opponents

In [ ]:
opponents = conn.execute("""
	SELECT
		opponent_abbr, total_shots, games_played, fg_pct,
		clutch_fg_pct, three_pt_rate, game_winning_shots,
		game_winning_made, home_fg_pct, away_fg_pct, rank_by_shots
	FROM main.mart_opponent_analysis
	GROUP BY
		opponent_abbr, total_shots, games_played, fg_pct,
		clutch_fg_pct, three_pt_rate, game_winning_shots,
		game_winning_made, home_fg_pct, away_fg_pct, rank_by_shots
	ORDER BY rank_by_shots
	LIMIT 10
""").fetchall()

print(f"{'Opp':<6} {'Shots':>6} {'Games':>6} {'FG%':>6} {'Clutch':>7} {'3PT%':>6} {'GW':>4} {'Home':>6} {'Away':>6}")
print('-' * 80)
for opp, shots, games, fg, cfg, r3, gw_s, gw_m, home, away, rank in opponents:
	cfg_str = f'{cfg}%' if cfg is not None else 'N/A'
	print(f'{opp:<6} {shots:>6,} {games:>6,} {fg:>6}% {cfg_str:>7} {r3:>5}% {gw_m or 0}/{gw_s or 0}  {home or "N/A":>5}% {away or "N/A":>5}%')

In [ ]:
print('Best FG% vs opponent (min 50 shots):')
best = conn.execute("""
	SELECT opponent_abbr, fg_pct, total_shots
	FROM main.mart_opponent_analysis
	GROUP BY opponent_abbr, fg_pct, total_shots
	HAVING total_shots >= 50
	ORDER BY fg_pct DESC
	LIMIT 5
""").fetchall()
for opp, fg, shots in best:
	print(f'  {opp:<6} {fg}%  ({shots:,} shots)')

print('\nWorst FG% vs opponent (min 50 shots):')
worst = conn.execute("""
	SELECT opponent_abbr, fg_pct, total_shots
	FROM main.mart_opponent_analysis
	GROUP BY opponent_abbr, fg_pct, total_shots
	HAVING total_shots >= 50
	ORDER BY fg_pct ASC
	LIMIT 5
""").fetchall()
for opp, fg, shots in worst:
	print(f'  {opp:<6} {fg}%  ({shots:,} shots)')

conn.close()